<a href="https://colab.research.google.com/github/szhang2187/Datathon_For_Social_Good_8/blob/data_cleaning_step_2/02_census_zip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Pull 14 Census numbers (income, population, race, no-car, limited-English)
# for every US ZIP into a table called acs. The API key comes from Colab Secrets.

import pandas as pd
import requests
from google.colab import userdata

key = userdata.get("Census_Key")

vars_ = ["B19013_001E", "B01003_001E",
         "B03002_001E", "B03002_003E", "B03002_004E", "B03002_006E", "B03002_012E",
         "B08201_001E", "B08201_002E",
         "C16002_001E", "C16002_004E", "C16002_007E", "C16002_010E", "C16002_013E"]

url = ("https://api.census.gov/data/2024/acs/acs5?get=NAME," + ",".join(vars_)
       + "&for=zip%20code%20tabulation%20area:*&key=" + key)

r = requests.get(url, timeout=120)
print(r.status_code, r.headers.get("Content-Type"))

data = r.json()
acs = pd.DataFrame(data[1:], columns=data[0])
print(acs.shape)
print(acs.columns.tolist())

200 application/json;charset=utf-8
(33772, 16)
['NAME', 'B19013_001E', 'B01003_001E', 'B03002_001E', 'B03002_003E', 'B03002_004E', 'B03002_006E', 'B03002_012E', 'B08201_001E', 'B08201_002E', 'C16002_001E', 'C16002_004E', 'C16002_007E', 'C16002_010E', 'C16002_013E', 'zip code tabulation area']


In [ ]:
# Keep Bay Area ZIPs (9 counties), convert Census text to numbers,
# set missing codes to NaN, drop ZIPs under 1,000 people.

rel_url = "https://www2.census.gov/geo/docs/maps-data/data/rel2020/zcta520/tab20_zcta520_county20_natl.txt"
bay = ["06001", "06013", "06041", "06055", "06075", "06081", "06085", "06095", "06097"]

rel = pd.read_csv(rel_url, sep="|", dtype=str)
rel["AREALAND_PART"] = pd.to_numeric(rel["AREALAND_PART"])
rel = rel.dropna(subset=["GEOID_ZCTA5_20"])
rel = rel.sort_values("AREALAND_PART", ascending=False).drop_duplicates("GEOID_ZCTA5_20")
rel = rel[rel["GEOID_COUNTY_20"].isin(bay)][["GEOID_ZCTA5_20", "GEOID_COUNTY_20"]]
rel.columns = ["zip", "county_fips"]

acs = acs.rename(columns={"zip code tabulation area": "zip"})
num = [c for c in acs.columns if c not in ("NAME", "zip")]
acs[num] = acs[num].apply(pd.to_numeric, errors="coerce")
acs[num] = acs[num].where(acs[num] >= 0)

zt = rel.merge(acs, on="zip", how="left")
print("Bay Area ZIPs:", len(zt))

before = len(zt)
zt = zt[zt["B01003_001E"] >= 1000]
print("dropped under 1,000 people:", before - len(zt))
print("ZIPs left:", len(zt))
print("income at the $250,001 cap:", (zt["B19013_001E"] >= 250001).sum())
print("income missing:", zt["B19013_001E"].isna().sum())

Bay Area ZIPs: 300
dropped under 1,000 people: 39
ZIPs left: 261
income at the $250,001 cap: 12
income missing: 2


In [ ]:
# Rename columns, compute shares, add income quintiles,
# county names, largest race group, and car-free / limited-English thirds.

zt = zt.rename(columns={"B19013_001E": "median_income", "B01003_001E": "population",
                        "B03002_001E": "race_total", "B03002_003E": "white", "B03002_004E": "black",
                        "B03002_006E": "asian", "B03002_012E": "hispanic",
                        "B08201_001E": "households", "B08201_002E": "no_car_households",
                        "C16002_001E": "lang_households"})

zt["limited_english_households"] = zt[["C16002_004E", "C16002_007E", "C16002_010E", "C16002_013E"]].sum(axis=1, min_count=1)
zt = zt.drop(columns=["C16002_004E", "C16002_007E", "C16002_010E", "C16002_013E"])

zt["pct_no_car"] = zt["no_car_households"] / zt["households"]
zt["pct_limited_english"] = zt["limited_english_households"] / zt["lang_households"]
for g in ["white", "black", "asian", "hispanic"]:
    zt["pct_" + g] = zt[g] / zt["race_total"]

pct_cols = ["pct_white", "pct_black", "pct_asian", "pct_hispanic"]
zt["largest_group"] = zt[pct_cols].idxmax(axis=1).str.replace("pct_", "")
zt.loc[zt[pct_cols].max(axis=1) < 0.5, "largest_group"] = "mixed"

zt["income_group"] = pd.qcut(zt["median_income"], 5, labels=["Q1", "Q2", "Q3", "Q4", "Q5"])
zt["car_free_group"] = pd.qcut(zt["pct_no_car"], 3, labels=["low", "middle", "high"])
zt["limited_english_group"] = pd.qcut(zt["pct_limited_english"], 3, labels=["low", "middle", "high"])

counties = {"06001": "Alameda", "06013": "Contra Costa", "06041": "Marin", "06055": "Napa",
            "06075": "San Francisco", "06081": "San Mateo", "06085": "Santa Clara",
            "06095": "Solano", "06097": "Sonoma"}
zt["county"] = zt["county_fips"].map(counties)

print(zt["income_group"].value_counts().sort_index())
print(zt["county"].value_counts())
print(zt["largest_group"].value_counts())
print(zt[["median_income", "pct_no_car", "pct_limited_english"]].describe().round(2))

income_group
Q1    52
Q2    52
Q3    51
Q4    52
Q5    52
Name: count, dtype: int64
county
Santa Clara      56
Alameda          47
Contra Costa     39
San Mateo        27
San Francisco    26
Sonoma           26
Marin            21
Solano           13
Napa              6
Name: count, dtype: int64
largest_group
mixed       120
white        94
asian        33
hispanic     14
Name: count, dtype: int64
       median_income  pct_no_car  pct_limited_english
count         259.00      261.00               261.00
mean       146963.97        0.09                 0.07
std         50229.92        0.11                 0.05
min         50129.00        0.00                 0.00
25%        107850.50        0.03                 0.03
50%        135635.00        0.05                 0.06
75%        183897.50        0.09                 0.10
max        250001.00        0.70                 0.28


In [ ]:
# Connect Google Drive to this notebook so it can read places_clean.csv and save zip_table.csv.

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Count services per ZIP (listings and distinct sites), compute rates per 10,000 residents, add median ratings, save zip_table.csv.

places = pd.read_csv("/content/drive/MyDrive/places_clean.csv", dtype={"zip": str})
places["type"] = places["service_type"].str.replace("clinic/hospital", "clinic").str.replace(" ", "_")
types = ["clinic", "bus_stop", "grocery", "school", "bank", "pharmacy", "food_bank"]

sites = places[places["is_site"]].pivot_table(index="zip", columns="type", values="place_id",
                                              aggfunc="count", fill_value=0).add_prefix("sites_")
listings = places.pivot_table(index="zip", columns="type", values="place_id",
                              aggfunc="count", fill_value=0).add_prefix("listings_")
ratings = places[places["enough_reviews"]].pivot_table(index="zip", columns="type", values="rating_clean",
                                                       aggfunc="median").add_prefix("median_rating_")

counts = sites.join(listings, how="outer").join(ratings, how="outer").reset_index()
zt = zt.merge(counts, on="zip", how="left")

count_cols = [c for c in zt.columns if c.startswith(("sites_", "listings_"))]
zt[count_cols] = zt[count_cols].fillna(0)

for t in types:
    zt["sites_per_10k_" + t] = zt["sites_" + t] / zt["population"] * 10000
    zt["has_none_" + t] = zt["sites_" + t] == 0

in_bay = places["zip"].isin(zt["zip"])
print("places in Bay Area table ZIPs:", in_bay.sum(), "of", len(places))
print("ZIPs with no pharmacy:", zt["has_none_pharmacy"].sum())
print(zt.groupby("income_group", observed=True)[["sites_per_10k_pharmacy", "sites_per_10k_grocery", "sites_per_10k_clinic"]].median().round(2))

zt.to_csv("/content/drive/MyDrive/datathon/data/processed/zip_table.csv", index=False)
print(zt.shape)

places in Bay Area table ZIPs: 82307 of 89455
ZIPs with no pharmacy: 56
              sites_per_10k_pharmacy  sites_per_10k_grocery  \
income_group                                                  
Q1                              0.93                   3.82   
Q2                              0.73                   3.25   
Q3                              0.89                   2.43   
Q4                              0.89                   2.44   
Q5                              0.72                   1.53   

              sites_per_10k_clinic  
income_group                        
Q1                           20.14  
Q2                           23.06  
Q3                           22.04  
Q4                           23.52  
Q5                           32.86  
(261, 59)
